# Đánh giá bộ câu hỏi — BATCH 2/3 (build_testset.py + run_eval.py) — Colab (D-182, D-183)

**Đây là 1 trong 3 file sinh đôi: `colab_runtime_eval_batch1.ipynb`,
`colab_runtime_eval_batch2.ipynb` (file này), `colab_runtime_eval_batch3.ipynb`
— KHÔNG phải một notebook tham số hoá.** D-183 (2026-09-05): không còn Colab Pro,
phiên Free ngắn hơn và không đảm bảo GPU liên tục; bộ 240 câu bị chia thành 3
batch 80 câu (`python -m src.test.split_testset`, chia round-robin theo `loai`
— mỗi batch có tỉ lệ văn_bản/hình/ngoài_phạm_vi gần giống bộ gốc, là một PHÂN
VÙNG thật, không câu nào trùng giữa 2 batch) để mỗi phiên ngắn hơn, và nếu một
phiên bị ngắt kết nối (pipeline D-182 KHÔNG resume được — xem mục "Mất một khả
năng" bên dưới) chỉ mất tối đa 80 câu chứ không mất cả 240.

**Cả 3 file đọc CHUNG một `database_png`/`testset_da_duyet` trên Drive**, nhưng
mỗi file ghi kết quả vào một thư mục Drive ĐÁNH SỐ riêng (`eval_results/batch1/`,
`batch2/`, `batch3/`) để 3 phiên — kể cả 2 phiên chạy SONG SONG bằng 2 tài khoản
Google khác nhau — không đè kết quả lên nhau. Sau khi đủ cả 3 thư mục, tải cả 3
về máy local rồi chạy `python -m src.test.merge_eval_batches` để gộp thành báo
cáo cuối cùng (240 câu) — xem `document/decision_log.html` mục D-183.

**File này (BATCH 2) KHÔNG chạy `retrieval_benchmark.py`** — bước đó không gọi
LLM (không cần chia batch) nên chỉ chạy ĐÚNG MỘT LẦN, ở file
`colab_runtime_eval_batch1.ipynb`. Mục 12 dưới đây bị bỏ qua có chủ đích, ĐỪNG
chạy lại — sẽ dựng lại `ablation_cache.json` lãng phí mà không cộng dồn được gì.

**PHẢI đã chạy `python -m src.test.split_testset` cục bộ trước** (sau khi
`draft.csv`/`meta.json` đã `--mark-reviewed`) và upload TOÀN BỘ thư mục
`testset_da_duyet/` lên Drive TRƯỚC khi Run all — thư mục đó phải có cả
`draft.csv`+`meta.json` gốc VÀ thư mục con `batches/` (`batch1.csv`/`batch2.csv`/
`batch3.csv` + `batches/meta.json`). Xem mục 7.

---

**Bấm Run all rồi đợi.** Notebook này KHÔNG chạy ETL gì cả — DB đã ở trạng thái
CUỐI CÙNG (D-162, `v4_formula_hybrid_fix` / `v19_pill_kernels`, 16.515 chunk), người
dùng đã tự tay upload TOÀN BỘ `database/` hiện có lên Drive (thư mục `database_png`,
cùng cấp `datasource_png`) — **không phải một checkpoint rút gọn kiểu ETL**.

**D-182 (2026-09-04) — pipeline test/eval viết lại từ đầu, đổi kiến trúc hoàn
toàn so với bản D-163..D-181 mà notebook này từng mô tả:**

1. **`src/test/testset/draft.csv` + `meta.json` PHẢI được sinh và duyệt tay
   TRÊN MÁY LOCAL trước, rồi upload lên Drive/Colab — KHÔNG sinh trên Colab.**
   `python -m src.test.build_testset` sinh nháp; `--mark-reviewed` cần nhập
   `xac-nhan-da-doc` từ bàn phím (input tương tác), không chạy được trên Colab
   không giám sát. `run_eval.py` (mục 14 dưới) **tự raise ngay lập tức** nếu
   `meta.json` chưa `human_reviewed: true` (`testset_common.require_human_reviewed`)
   — không có đường vòng, và việc này chỉ lộ ra SAU khi model đã tải xong (mục
   4), nên upload đúng `draft.csv`/`meta.json`/`batches/` đã duyệt+chia LÀ VIỆC
   ĐẦU TIÊN phải làm trước khi bấm Run all, đừng để tới lúc đó mới phát hiện thiếu.
2. `run_eval.py` — pipeline RAG thật (retrieval → Qwen2.5-3B sinh câu trả lời →
   LLM thứ hai chấm điểm) cho **TOÀN BỘ câu hỏi trong MỘT file CSV** — không
   còn khái niệm "theo quyển", không còn vòng lặp 13 mục, không còn
   `--book`/`--bo-qua-da-co`. Thay `evaluator.py` (đã xoá). **D-183: file này
   truyền `--testset-csv` trỏ vào batch của nó, KHÔNG phải `draft.csv` đầy đủ.**

**Mất một khả năng so với bản cũ, CHƯA khôi phục lại (finding I-4, PARK có chủ
đích khi triển khai D-182):** `run_eval.py` KHÔNG có resume/checkpoint giữa
chừng — nếu Colab bị ngắt kết nối giữa lúc chạy, phải chạy lại **TỪ ĐẦU của batch
đó**, không resume được từng câu đã tính. Notebook này giữ nguyên "giữ phiên
sống" (mục 13) để giảm rủi ro bị ngắt, nhưng không có gì cứu được nếu vẫn xảy
ra — đây chính là lý do D-183 chia nhỏ thành 3×80 thay vì chạy 240 câu một lượt.

## Vì sao chạy trên Colab thay vì máy dev, và vì sao KHÔNG tải zip về

Máy dev có GPU **GTX 1050 Ti (4 GB VRAM)** — đo được bước sinh câu trả lời tốn
**~3–3,5 phút/câu** (D-164). 240 câu ước tính **12-14 giờ** trên máy dev; Colab
(kể cả tier Free) có GPU khá hơn nên nhanh hơn nhiều cho phần Qwen2.5-3B — đó là
khâu chiếm hầu hết thời gian, KHÔNG phải bước gọi Groq (Groq trả lời dưới 1
giây/lượt, đo D-163). **Chưa có phép đo phút/câu THẬT trên Colab** (D-183) — đây
là lý do bổ sung để ưu tiên batch nhỏ (80) thay vì batch lớn (120): rủi ro mất
trắng một phiên nếu ước lượng sai vẫn thấp hơn.

**Mọi kết quả được đồng bộ THẲNG lên Drive** (`EVAL_RESULTS_DRIVE_DIR` ở mục 10),
không dùng `files.download()`. Cell cuối chỉ in đường dẫn Drive, không tải gì về
máy qua trình duyệt.

## LLM giám khảo: Groq, BỐN model xoay vòng (D-163, D-173)

`stealth/ox-alpha` (OpenRouter) đã hết free. Nay dùng Groq:
`qwen/qwen3.8-27b` + `openai/gpt-oss-120b` + `qwen/qwen3.6-27b` + `openai/gpt-oss-20b`,
xoay vòng qua `JudgePool` (`src/test/llm_client.py`, đổi tên từ `eval_llm.py` ở
D-182) khi một model bị rate-limit.
**D-173:** lượt chạy 240 câu 2026-09-02 với chỉ HAI model dính hạn mức **TPD
(200 000 token/ngày/model)** giữa chừng — khác với TPM (8 000 token/phút/model) đã
đo D-163 — vì cả hai model cùng cạn cùng lúc nên `JudgePool` xoay qua xoay lại vô
ích, 106/240 câu (44%) mất điểm judge (`LỖI: ... tokens per day (TPD) ...`). Thêm
2 model để nhân đôi ngân sách/ngày (mỗi model một bucket riêng) — không sửa code,
`JudgePool` đã xoay N-chiều tổng quát. **D-183: chia 3 batch cũng làm giảm nhịp
gọi Groq mỗi phiên, giảm thêm rủi ro chạm TPD so với dồn cả 240 câu vào 1 phiên.**
**Khoá Groq lấy từ Colab Secrets, KHÔNG gõ thẳng vào notebook** (notebook này
được commit vào git).

## 1. Clone repo

In [ ]:
!git clone -b master https://github.com/lcdkhoa/project-bio-rag.git
%cd project-bio-rag

In [ ]:
!git log --oneline -3

## 1b. KHÔNG CÒN CẦN THIẾT (D-182) — giữ mục trống để không đảo số các mục sau

Bài học cũ (git clone mang theo `*_result.csv` cũ, khiến `--bo-qua-da-co` tưởng
nhầm là đã xong) không còn áp dụng: `src/test/testset/` (nơi `draft.csv`/
`eval_result.csv`/... sống) đã bị `.gitignore` loại từ D-182 — `git clone` ở mục 1
KHÔNG mang theo bất kỳ kết quả cũ nào nữa. Ô dưới chỉ còn là no-op, giữ lại để số
thứ tự các mục phía sau không phải đổi hàng loạt.

In [ ]:
print("Bo qua (D-182) - src/test/testset/ da bi .gitignore loai, git clone khong con mang theo ket qua cu.")

## 2. Cài dependencies

Không cần `mineru_vl_utils`/pin `transformers` như notebook ETL — đường đánh giá
không gọi MinerU (bước OCR công thức chỉ chạy lúc ETL, đã xong). Cũng không cần
`poppler`/`tesseract` (không OCR trang nào ở đây).

In [ ]:
!pip install -q -r requirements.txt
import transformers
print("transformers:", transformers.__version__)

## 3. Secrets (đặt TRƯỚC khi tải model)

Mở tab 🔑 (Secrets) bên trái, thêm HAI khoá, bật *Notebook access* cho cả hai:

- `HF_TOKEN` — tải model từ HuggingFace (Qwen2.5-3B-Instruct, bge-m3, reranker, CLIP).
- `GROQ_API_KEY` — LLM giám khảo (D-163). **Không dán khoá thẳng vào ô code** — notebook
  này commit vào git công khai được, một khoá lộ trong đó là khoá phải thu hồi.

In [ ]:
import os, multiprocessing
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

n = multiprocessing.cpu_count()
os.environ["OMP_NUM_THREADS"] = str(n)
os.environ["NUMEXPR_NUM_THREADS"] = str(n)
os.environ["OPENBLAS_NUM_THREADS"] = str(n)
os.environ["USE_GPU"] = "true"
print("OK, cpu count:", n)

## 4. Tải model về `./models` (chạy ONLINE, trước khi bật offline)

Dùng profile `serve` (`src/utils/download_models.py`) — đúng bốn model cần cho
truy vấn + sinh câu trả lời: `bge-m3` (embedding), `bge-reranker-v2-m3` (rerank),
`Qwen2.5-3B-Instruct` (sinh câu trả lời), `clip-vit-base-patch16` (ảnh — AppServices
nạp cả collection ảnh dù bộ test này chỉ có câu hỏi văn bản/đã có gold key text).

In [ ]:
import subprocess, sys

r = subprocess.run([sys.executable, "-u", "./src/utils/download_models.py",
                    "--save_dir", "./models", "--profile", "serve"])
if r.returncode != 0:
    raise RuntimeError(
        f"Tai model that bai (ma thoat {r.returncode}) - DUNG o day.")
print("Tai model xong, ma thoat 0.")

## 5. Mount Drive + đường dẫn

**KHÔNG có checkpoint kiểu ETL ở đây** — người dùng đã tự tay upload TOÀN BỘ
`database/` hiện có lên Drive, đặt tên `database_png` (song song `datasource_png`
của notebook ETL). Trong `database/images/<quyển>/`, thư mục con `snapshot/` đã bị
XOÁ TAY để giảm dung lượng — chỉ còn hình đã cắt; không ảnh hưởng bước đánh giá này
(`run_eval.py` không đọc file ảnh, chỉ đọc `biology_text`/BM25/`processing_status`).

**Sửa `DB_SOURCE_DIR` dưới đây nếu tên/đường dẫn thư mục trên Drive của bạn khác** —
đây là suy đoán theo quy ước đặt tên `datasource_png`, CHƯA được xác minh trực tiếp
trên Drive thật của bạn.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

BATCH_INDEX = 2  # File nay CO DINH batch 2/3 (D-183) - KHONG doi, mo dung file cho batch ban dinh chay

DB_SOURCE_DIR = "/content/drive/MyDrive/project_bio_rag/database_png"  # SUA neu khac
TESTSET_SOURCE_DIR = "/content/drive/MyDrive/project_bio_rag/testset_da_duyet"  # D-182/D-183: draft.csv+meta.json+batches/ da duyet+chia tay TREN MAY LOCAL, upload len day truoc
os.environ["RAG_DATABASE_DIR"] = "/content/database"  # dia cuc bo, tranh Drive-FUSE cho SQLite (D-152)
EVAL_RESULTS_DRIVE_DIR = f"/content/drive/MyDrive/project_bio_rag/eval_results/batch{BATCH_INDEX}"  # D-183: danh so theo batch, 3 phien (ke ca 2 phien song song) khong de len nhau
BATCH_CSV = f"src/test/testset/batches/batch{BATCH_INDEX}.csv"  # D-183: input cho run_eval.py (muc 14)

base = "/content/project-bio-rag/models"
os.environ["EMBEDDING_MODEL"] = f"{base}/bge-m3"
os.environ["RERANK_MODEL"] = f"{base}/bge-reranker-v2-m3"
os.environ["LLM_MODEL"] = f"{base}/Qwen2.5-3B-Instruct"
os.environ["CLIP_MODEL"] = f"{base}/clip-vit-base-patch16"
os.environ["HF_HUB_OFFLINE"] = "1"  # model da tai o muc 4

for key in ("RAG_DATABASE_DIR", "EMBEDDING_MODEL", "RERANK_MODEL", "LLM_MODEL",
            "CLIP_MODEL", "HF_HUB_OFFLINE"):
    print(f"{key} = {os.environ[key]}")
print("DB_SOURCE_DIR =", DB_SOURCE_DIR)
print("TESTSET_SOURCE_DIR =", TESTSET_SOURCE_DIR)
print("BATCH_INDEX =", BATCH_INDEX, "-> BATCH_CSV =", BATCH_CSV)
print("EVAL_RESULTS_DRIVE_DIR =", EVAL_RESULTS_DRIVE_DIR)

## 6. Hàm copy chịu Drive-FUSE rớt kết nối (giống `colab_runtime_etl.ipynb`, D-161)

Dùng cho CẢ HAI hướng: đọc `database_png` xuống đĩa cục bộ, và ghi kết quả lên
`EVAL_RESULTS_DRIVE_DIR`.

In [ ]:
import shutil
import time
from pathlib import Path


def _copy_resilient(src: Path, dst: Path, tries: int = 5, delay: float = 5.0):
    """Di tung file, thu lai khi Drive-FUSE rot ket noi (ENOTCONN, D-161) - mot
    file loi khong huy phan cay da sao chep duoc."""
    if src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for child in src.iterdir():
            _copy_resilient(child, dst / child.name, tries, delay)
        return
    for attempt in range(1, tries + 1):
        try:
            shutil.copy2(src, dst)
            return
        except OSError as exc:
            if attempt == tries:
                raise
            print(f"  loi copy {src.name} (lan {attempt}/{tries}): {exc} -> thu lai sau {delay}s")
            time.sleep(delay)


print("OK, dinh nghia xong _copy_resilient")

## 7. Khôi phục DB (CHỈ ĐỌC từ `database_png`) + bộ test đã duyệt + chia batch

Copy TOÀN BỘ `DB_SOURCE_DIR` vào đĩa cục bộ — không có logic loại trừ/checkpoint
từng phần như ETL, vì đây là bản upload MỘT LẦN, đã ở trạng thái cuối.

**D-182: cũng copy `draft.csv`+`meta.json` đã duyệt tay TRÊN MÁY LOCAL** từ
`TESTSET_SOURCE_DIR` trên Drive — notebook này KHÔNG tự sinh bộ test.
**D-183: cũng copy thư mục con `batches/`** (`batch1.csv`/`batch2.csv`/
`batch3.csv` + `batches/meta.json`, sinh bằng `python -m src.test.split_testset`
cục bộ) — `run_eval.py` (mục 14) đọc `BATCH_CSV` (batch2) trong đó. Upload cả
cây `testset_da_duyet/` lên Drive TRƯỚC khi Run all, đúng đường dẫn
`TESTSET_SOURCE_DIR` ở mục 5 (sửa nếu khác).

In [ ]:
src_dir = Path(DB_SOURCE_DIR)
if not src_dir.exists():
    parent = src_dir.parent
    goi_y = list(parent.iterdir()) if parent.exists() else []
    raise RuntimeError(
        f"Khong thay {src_dir}. Cac thu muc con trong {parent}:\n"
        + "\n".join(str(p) for p in goi_y)
        + "\n-> sua DB_SOURCE_DIR o muc 5 cho dung, dung doan.")

local_db = Path(os.environ["RAG_DATABASE_DIR"])
local_db.mkdir(parents=True, exist_ok=True)
for item in src_dir.iterdir():
    print("dang khoi phuc:", item.name)
    _copy_resilient(item, local_db / item.name)
print("XONG khoi phuc DB tu Drive.")

# D-182: copy draft.csv + meta.json da duyet tay tren may local ve dung vi tri
# src/test/testset/ (khong dung o file nay - batch2 khong chay retrieval_benchmark.py
# - nhung khoi phuc cho dong nhat voi batch1/batch3, khong hai gi).
# D-183: cung copy thu muc con batches/ (batch1.csv/batch2.csv/batch3.csv +
# batches/meta.json) - run_eval.py (muc 14) doc BATCH_CSV trong do.
testset_src = Path(TESTSET_SOURCE_DIR)
if not testset_src.exists():
    raise RuntimeError(
        f"Khong thay {testset_src}. Ban PHAI sinh + duyet tay draft.csv/meta.json "
        "TREN MAY LOCAL (python -m src.test.build_testset roi --mark-reviewed), "
        "chia batch (python -m src.test.split_testset), roi upload TOAN BO thu "
        "muc nay len dung TESTSET_SOURCE_DIR TRUOC khi Run all - notebook nay "
        "KHONG tu sinh/chia bo test (D-182, D-183).")
local_testset = Path("src/test/testset")
local_testset.mkdir(parents=True, exist_ok=True)
for name in ("draft.csv", "meta.json"):
    p = testset_src / name
    if not p.exists():
        raise RuntimeError(f"Thieu {p} trong TESTSET_SOURCE_DIR.")
    _copy_resilient(p, local_testset / name)

batches_src = testset_src / "batches"
if not batches_src.exists():
    raise RuntimeError(
        f"Thieu {batches_src} trong TESTSET_SOURCE_DIR - chay "
        "`python -m src.test.split_testset` cuc bo roi upload lai (D-183).")
_copy_resilient(batches_src, local_testset / "batches")
if not Path(BATCH_CSV).exists():
    raise RuntimeError(
        f"Thieu {BATCH_CSV} sau khi khoi phuc batches/ - kiem tra split_testset.py "
        "da chay dung chua (co du 3 file batch1/2/3.csv khong).")
print("XONG khoi phuc DB + draft.csv + meta.json + batches/ tu Drive.")

## 8. Xác nhận index khôi phục đúng — ĐỪNG bỏ qua bước này

Đo trực tiếp trên chính DB vừa khôi phục, không tin tên thư mục Drive. Kỳ vọng
(khớp D-162, đo trên máy dev cùng ngày): `processing_status` 2399/2399 trang ở
`text_extraction_version = v4_formula_hybrid_fix`; `biology_text` **16.515** chunk;
BM25 (`database/sparse/bm25_meta.json`) **16.515** id.

In [ ]:
import json
import sys
from collections import Counter

sys.path.insert(0, "/content/project-bio-rag")
import chromadb

client = chromadb.PersistentClient(path=os.environ["RAG_DATABASE_DIR"])

ps = client.get_collection("processing_status")
docs = [json.loads(d) for d in ps.get(include=["documents"])["documents"]]
print("processing_status:", len(docs))
print("text_extraction_version:", Counter(d.get("text_extraction_version") for d in docs))

bt = client.get_collection("biology_text")
print("biology_text count:", bt.count())

bm25_meta_path = Path(os.environ["RAG_DATABASE_DIR"]) / "sparse" / "bm25_meta.json"
if bm25_meta_path.exists():
    with open(bm25_meta_path, encoding="utf-8") as f:
        meta = json.load(f)
    print("BM25 ids:", len(meta["ids"]), "| vocab:", len(meta["vocab"]))
else:
    print("CANH BAO: khong thay", bm25_meta_path, "- BM25/hybrid se khong chay duoc.")

assert bt.count() == 16515, f"So chunk khong khop ky vong D-162 (16515), do duoc {bt.count()} - DUNG, kiem tra lai DB_SOURCE_DIR."
print("\nOK - index khop ky vong D-162, chay tiep duoc.")

## 9. Cấu hình LLM giám khảo (Groq, D-163)

In [ ]:
os.environ["EVAL_LLM_BASE_URL"] = "https://api.groq.com/openai/v1"
os.environ["EVAL_LLM_API_KEY"] = os.environ["GROQ_API_KEY"]
os.environ["EVAL_LLM_MODEL"] = "qwen/qwen3.8-27b"
os.environ["EVAL_LLM_MODELS"] = "qwen/qwen3.8-27b,openai/gpt-oss-120b,qwen/qwen3.6-27b,openai/gpt-oss-20b"

from src.test.llm_client import get_eval_llm, is_configured
print("configured:", is_configured())
_llm = get_eval_llm(temperature=0.0)
_resp = _llm.invoke("Tra loi dung mot chu: OK")
print("smoke test:", _resp.content)

## 10. Hàm đồng bộ kết quả lên Drive

**D-182: đơn giản hoá đáng kể so với bản D-163..D-181** — không còn 13 file
theo quyển, không còn resume-per-book (`khoi_phuc_tien_do_da_co` cũ đã xoá).
`retrieval_benchmark.py`/`run_eval.py` ghi kết quả vào MỘT bộ file duy nhất ở
`src/test/testset/` (`retrieval_report.*`, `eval_result.csv`, `eval_report.*`).

**Không có resume nếu Colab bị ngắt giữa lúc `run_eval.py` đang chạy** (finding
I-4, PARK — xem mục 0). `dong_bo_ket_qua()` chỉ đẩy TRẠNG THÁI CUỐI (sau khi một
lệnh chạy xong hoàn toàn) lên Drive — gọi nó SAU mỗi lệnh `subprocess.run`, không
gọi giữa chừng vì không có "giữa chừng" để đồng bộ.

In [ ]:
drive_out = Path(EVAL_RESULTS_DRIVE_DIR)
local_testset = Path("src/test/testset")


def dong_bo_ket_qua():
    """D-182: dong bo TOAN BO src/test/testset/ len Drive - mot thu muc phang,
    khong con phan biet 'file tong hop' vs 'file tung quyen' nhu ban cu."""
    drive_out.mkdir(parents=True, exist_ok=True)
    for p in local_testset.iterdir():
        if p.is_file():
            _copy_resilient(p, drive_out / p.name)
    print("Da dong bo ket qua ->", drive_out)


print("OK, dinh nghia xong dong_bo_ket_qua() (D-182, khong con resume-per-book).")

## 11. `ablation.py` đã bị xóa hoàn toàn (D-182) — thay bằng `retrieval_benchmark.py`, xem mục dưới

`recall_at_k.py` đã gộp vào `ablation.py` từ D-181, và cả `ablation.py` chính nó
đã bị xóa hoàn toàn ở D-182 — logic MRR/K∈{1,3,5,10,20} + bảng "Bề rộng
PRODUCTION" được giữ nguyên và chuyển sang `retrieval_benchmark.py` (mục 12).

In [ ]:
print("Bo qua - ablation.py da xoa hoan toan (D-182), xem muc 12 (retrieval_benchmark.py).")

## 12. `retrieval_benchmark.py` — BỎ QUA ở batch này (D-183)

Bước này (bảng 12 cấu hình, không gọi LLM) chỉ chạy ĐÚNG MỘT LẦN, ở file
`colab_runtime_eval_batch1.ipynb` — không cần chia batch vì không tốn quota Groq.
Chạy lại ở đây chỉ dựng lại `ablation_cache.json` một cách lãng phí (tốn CPU/GPU
embedding+rerank lần nữa) mà không cộng dồn thêm được gì, vì bảng đó cần TOÀN BỘ
240 câu chứ không phải batch 80 câu của file này.

In [ ]:
print("Bo qua (D-183) - retrieval_benchmark.py chi chay o colab_runtime_eval_batch1.ipynb.")

## 13. Giữ phiên sống trước khi chạy bước dài

In [ ]:
%%javascript
function KeepClicking(){
  var btn = document.querySelector("colab-connect-button");
  if (btn) { btn.click(); console.log("Da bam connect luc " + new Date()); }
}
setInterval(KeepClicking, 60000);

## 14. Bước 3 — `run_eval.py` cho BATCH này (mục chính, TỐN THỜI GIAN NHẤT) (D-182, D-183, thay `evaluator.py`)

**D-182: KHÔNG còn vòng lặp 13 mục theo quyển.** MỘT lệnh `run_eval.py` duy nhất
chạy trên `BATCH_CSV` (80 câu của batch này, KHÔNG phải `draft.csv` đầy đủ —
xem D-183 ở mục 0) — không còn `BOOKS`, không còn `--book`/`--bo-qua-da-co`.
`run_eval.py` tự in tiến trình `[i/N]` cho từng câu (N=80 ở đây, không phải 240).

**Không có resume nếu bị ngắt giữa chừng (finding I-4, PARK — xem mục 0).** Một
phiên Colab bị ngắt kết nối giữa lúc ô dưới đang chạy nghĩa là phải chạy lại
TỪ ĐẦU CỦA BATCH NÀY (80 câu, không phải 240 — đây là lý do D-183 chia batch) —
mục 13 (giữ phiên sống) giảm rủi ro này nhưng không loại bỏ hoàn toàn. Chỉ đồng
bộ Drive SAU KHI lệnh chạy xong hoàn toàn (không có "giữa chừng" an toàn để
đồng bộ như bản cũ).

In [ ]:
r = subprocess.run([sys.executable, "-u", "-m", "src.test.run_eval",
                    "--testset-csv", BATCH_CSV])
print("exit code:", r.returncode)
if r.returncode != 0:
    raise RuntimeError(
        f"run_eval.py THAT BAI (ma {r.returncode}) - DUNG, xem log truoc khi chay "
        f"lai (khong resume duoc, finding I-4 PARK - xem muc 0). "
        f"BATCH_INDEX={BATCH_INDEX}, BATCH_CSV={BATCH_CSV}")
dong_bo_ket_qua()
print(f"--- run_eval.py (BATCH {BATCH_INDEX}) da xong, da dong bo len {EVAL_RESULTS_DRIVE_DIR} ---")

## 15. Xác nhận sanity — số câu trong `eval_result.csv` khớp `BATCH_CSV` (batch này)

**D-182: đơn giản hoá hoàn toàn so với cổng mtime-theo-BOOKS cũ** (D-168, từng
xử lý 13 file resume riêng lẻ) — không còn cần thiết vì `run_eval.py` chạy một
lần, không resume (finding I-4, PARK). **D-183: so với `BATCH_CSV` (80 câu),
KHÔNG phải `draft.csv` đầy đủ (240 câu)** — batch này chỉ chấm đúng phần của nó.
Kiểm tra: số dòng `eval_result.csv` phải khớp đúng số dòng `BATCH_CSV`.

In [ ]:
def _so_dong_csv(p):
    # C-B (phan bien Opus 5, 2026-09-04): DEM DONG THO (`sum(1 for _ in f)`) la
    # SAI khi cot rag_answer/ground_truth/judge_reasoning chua xuong dong VAT
    # LY ben trong o da duoc CSV quote - mot ban ghi co the trai nhieu dong vat
    # ly, lam so dem RA LON HON so ban ghi that. Phai dung csv.reader (khong
    # phai dem dong tho) vi cau tra loi/reasoning co the chua xuong dong vat ly
    # trong o da quote - dem dong tho se sai.
    import csv
    with open(p, encoding="utf-8-sig", newline="") as f:
        return sum(1 for _ in csv.reader(f)) - 1  # tru dong header


so_cau_batch = _so_dong_csv(Path(BATCH_CSV))
so_cau_result = _so_dong_csv(local_testset / "eval_result.csv")
print(f"{BATCH_CSV}: {so_cau_batch} cau | eval_result.csv: {so_cau_result} dong")
if so_cau_result != so_cau_batch:
    raise RuntimeError(
        f"LECH SO CAU: {BATCH_CSV} co {so_cau_batch} cau nhung eval_result.csv "
        f"chi co {so_cau_result} dong - CHUA HOP LE, dung tin bang tong hop.")
print(f"OK - so cau khop cho BATCH {BATCH_INDEX}, moi cau trong batch nay deu co dong ket qua tuong ung.")

## 16. Xem lại bảng tổng hợp — RIÊNG BATCH NÀY, chưa phải số cuối cùng (D-183)

Số dưới đây chỉ tính trên 80 câu của batch này (không tải gì về máy — đã nằm
trên Drive). Sau khi CẢ 3 BATCH đều xong: tải cả 3 thư mục `eval_results/batch1`,
`batch2`, `batch3` từ Drive về máy local, rồi chạy
`python -m src.test.merge_eval_batches` để có báo cáo cuối cùng trên 240 câu.

In [ ]:
import pandas as pd
d = pd.read_csv("src/test/testset/eval_report.csv")
print(f"=== Bao cao RIENG cua BATCH {BATCH_INDEX}/3 - CHUA phai so cuoi cung ===")
print(d.to_string())
print()
print("So cau trong batch nay:", int(d["num_questions"].sum()),
      "(khop so dong da xac nhan o muc 15; day la mot phan cua 240 cau)")
print("\nKet qua batch nay da nam o:", EVAL_RESULTS_DRIVE_DIR)
print("\nSau khi CA 3 BATCH deu xong: tai ca 3 thu muc Drive")
print("(eval_results/batch1, batch2, batch3) ve may local, roi chay:")
print("  python -m src.test.merge_eval_batches")
print("de co bao cao cuoi cung tren 240 cau.")